# Gemini LLM API

Python 코드에서 LLM에게 요청을 보내고 응답을 사용하는 기본 흐름을 익힙니다.

공식 문서: https://ai.google.dev/gemini-api/docs/get-started

## 실습 환경 준비

패키지를 설치합니다. 설치 후 import 오류가 계속되면 커널을 한 번 재시작하세요.

`pip install google-genai`

### API 키 준비

프로젝트 폴더의 `.env` 파일에 다음과 같이 적습니다.

```text
GEMINI_API_KEY=발급받은_API_키
```

API 키는 비밀번호와 같습니다. 코드에 직접 적거나 Git에 올리지 마세요. `.gitignore`에 `.env`가 포함되어 있는지도 확인합니다.

In [1]:
import os
from pprint import pprint

import requests
from dotenv import load_dotenv
from google import genai
from google.genai import errors

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")

# 모델 이름은 바뀔 수 있으므로 환경 변수로 교체할 수 있게 둡니다.
model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
print("준비 완료 / 사용 모델:", model)


준비 완료 / 사용 모델: gemini-3.6-flash


## REST API로 첫 요청 보내기

SDK를 사용하기 전에 HTTP 요청의 구성 요소를 직접 확인합니다. `requests.post()`가 URL로 요청을 보내고, headers에는 인증 정보, body에는 모델이 처리할 데이터를 담습니다.

- URL: 요청을 받을 API 주소
- headers: 데이터 형식과 API 키
- body: 모델, 입력, 요청 설정
- response: 상태 코드와 응답 body

In [2]:
url = "https://generativelanguage.googleapis.com/v1beta/interactions"
headers = {
    "Content-Type": "application/json",
    "x-goog-api-key": api_key,
}
body = {
    "model": model,
    "input": "LLM API를 30자 이내의 한 문장으로 설명해줘.",
    "store": False,
}

response = requests.post(url, headers=headers, json=body, timeout=30)
print("HTTP 상태 코드:", response.status_code)
response.raise_for_status()

response_data = response.json()
pprint(response_data)

# 기본 텍스트 요청에서는 마지막 step에 모델의 최종 답변이 들어 있습니다.
answer = response_data["steps"][-1]["content"][0]["text"]
print("\n최종 답변:")
print(answer)

HTTP 상태 코드: 200
{'created': '2026-08-25T05:31:14Z',
 'model': 'gemini-3.6-flash',
 'object': 'interaction',
 'service_tier': 'standard',
 'status': 'completed',
 'steps': [{'signature': 'EpwsCpksARFNMg+o5jLvKok5NugJycM/NvWdADev/yG6acxZdBoXjtc1JjikiWovG+Ra7sEZUaU3fD+Z7MBRrnv1dSBDWQTMEgoV71XX6kNxHAYLid1/kPn4WM5dDG1HI4OhZ2g7qlCKHYmpiZBdLytCSVolkNBJBNIKHr6yWuszGt+jfaDtQiFC4WxOt0/YV5CVbdKCloeHWdVMllwu86vdw3feqFNJmgzt4l0EfVbpyDcrKJfDjApxjhnNjdTEcGAe5LKVsggxpWWErinJt2nMjjRT+Dnh+FcYTbaEACZqxzl0MhHZE40Y1euQzpT8T2gOW1FUu8lEi+GLX6k72HLoJLuZitpeRyAMRf4ATboHrfTnqUbjtUh/dPBfTdskfBDVx0hD/fC/cZIVFX+F5wQfeR8UY8WzzdAx4DcMOUGTAI5fbrU+0aAAgJBlu/ByfdwfsL7oId0kQ+mYLCvWra0DH+7+PVjLSCQ5WKqdLq83SeSKNug9PPHtxE0lKhlmALti4DNp/KGPUviCWf2myk+wDH8KlOaouQtuzjRMUMMG+Z88p10fvliCNIOh8KmY/A+r8hCN66NmI7aG4EUGaQa6V5o061K3sv2CZaerHlFzgYCxD9mLpfEsFZq6NnkUf5HCiahUksON1Ta4NPPoGPV8vySMUgSGNYr9ABYEMQNZBBs8sX/4QL1rrepFIeWrXolb+6faT0qbq0QBSur2LBP6evlStkNqU4hob0QtHCfYMRrovaLA0g/qLoBXtfAmUg4lYBX1pMx9+RuSsRkTI/5abuGRHWnBTJjrvB0GFROBq

REST 응답은 대략 다음과 같은 구조입니다. ID와 답변 내용은 실행할 때마다 달라질 수 있습니다.

```python
{
    "id": "interaction의 고유 ID",
    "status": "completed",
    "steps": [
        {"type": "user_input", "content": [...]},
        {
            "type": "model_output",
            "content": [
                {"type": "text", "text": "LLM API는 ..."}
            ],
        },
    ],
}
```


## SDK로 같은 요청 보내기

SDK는 앞에서 직접 작성한 URL, headers, JSON 변환과 응답 구조 처리를 편리하게 감싸 줍니다. HTTP 요청이 사라진 것이 아니라 라이브러리 안에서 대신 처리되는 것입니다.

- `model`: 사용할 모델
- `input`: 모델이 처리할 질문이나 데이터
- `output_text`: SDK가 최종 텍스트를 꺼내 제공하는 편의 속성

In [3]:
client = genai.Client(api_key=api_key)

interaction = client.interactions.create(
    model=model,
    input="LLM API를 30자 이내의 한 문장으로 설명해줘.",
    store=False,
)

print(interaction.output_text)


AI 언어 모델을 앱에 연결해 주는 도구입니다.


### SDK 코드와 응답 읽기

- `genai.Client(...)`: API 키를 가진 Gemini 클라이언트 객체를 만듭니다. 이후 요청은 이 객체를 통해 보냅니다.
- `client.interactions.create(...)`: 새 interaction 요청을 만들어 Gemini API로 전송합니다. 이름은 `create`이지만 로컬 객체만 만드는 것이 아니라 실제 네트워크 요청이 일어납니다.
- `interaction.output_text`: 여러 step으로 구성된 응답에서 최종 텍스트를 SDK가 꺼내 제공하는 편의 속성입니다.

`create()`가 반환하는 `Interaction` 객체는 개념적으로 다음 정보를 가집니다. 실제 객체를 dictionary처럼 대괄호로 읽는 것은 아니고 점(`.`)으로 속성에 접근합니다.

```python
interaction.id           # 요청의 고유 ID
interaction.status       # completed, failed 등의 처리 상태
interaction.steps        # 사용자 입력과 모델 출력 과정
interaction.output_text  # 마지막 모델 출력의 최종 텍스트
```

REST의 JSON 응답과 SDK의 `Interaction` 객체는 서로 다른 결과가 아니라, 같은 API 응답을 서로 다른 방식으로 읽는 것입니다.

### `input`에 넣을 수 있는 두 가지 형태

간단한 텍스트 요청에서는 `input`에 문자열 하나를 바로 넣을 수 있습니다.

```python
input="안녕하세요"
```

대화 기록처럼 입력의 역할과 종류를 구분해야 할 때는 dictionary로 만든 step의 list를 넣습니다.

```python
input=[
    {
        "type": "user_input",
        "content": [
            {"type": "text", "text": "안녕하세요"}
        ],
    }
]
```

여기서 `input` 자체가 `type`으로 바뀌는 것은 아닙니다. `input`에 들어가는 각 항목이 `type` 필드로 자신의 역할을 표시합니다. `user_input`은 사용자 입력 step, 그 안의 `text`는 텍스트 콘텐츠라는 뜻입니다.

| 상황 | `input` 형태 |
|---|---|
| 한 번의 간단한 질문 | 문자열 `str` |
| 텍스트 외 다른 종류를 구분 | `type`이 있는 dictionary의 `list` |

### 직접 바꿔 보기

질문의 주제, 답변 문장 수, 설명을 듣는 대상을 바꿔 다시 실행해 보세요.

**생각해 볼 질문:** 같은 질문을 다시 보내면 답변이 완전히 같나요? 다른 API들과는 무엇이 다른가요?

## System instruction과 사용자 입력

`system_instruction`은 답변 전반에 적용할 역할과 규칙이고, `input`은 이번에 처리할 실제 질문입니다. 역할, 작업, 제약, 출력 형식을 구체적으로 적으면 의도를 전달하기 쉽습니다.

> 프롬프트는 답변 품질을 유도하지만 사실성을 보장하지는 않습니다. 중요한 결과는 별도로 검증합니다.

In [ ]:
question = "API 키를 코드에 직접 적으면 왜 위험해? 한 문장으로 답해줘."
instructions = {
    "친절한 강사": "Python 입문 강사입니다. 쉬운 말로 40자 이내에서 답하세요.",
    "보안 담당자": "보안 담당자입니다. 단호한 말투로 40자 이내에서 답하세요.",
}

for name, instruction in instructions.items():
    result = client.interactions.create(
        model=model, input=question,
        system_instruction=instruction, store=False,
    )
    print(f"--- {name} ---")
    print(result.output_text, "\n")

## 대화 맥락 이어가기



첫 요청에서 이름을 알려줘도, 두 번째 요청에 이전 대화를 넣지 않으면 모델은 이름을 알 수 없습니다.

In [4]:
# 첫 번째 요청: 이름 알려주기
tell_name = client.interactions.create(
    model=model,
    input="내 이름은 민수야. '확인'이라고만 답해.",
    store=False,
)
print("첫 번째 답변:", tell_name.output_text)

# 두 번째 요청: 이전 대화를 전달하지 않고 이름 물어보기
ask_without_history = client.interactions.create(
    model=model,
    input="내 이름이 뭐야? 정보가 없으면 '모름'이라고만 답해.",
    store=False,
)
print("두 번째 답변:", ask_without_history.output_text)

첫 번째 답변: 확인
두 번째 답변: 모름


### history를 넣어 두 번 요청하기

이번에는 첫 요청의 입력과 응답을 history에 보존한 뒤 두 번째 질문과 함께 전달합니다.

In [ ]:
# 첫 번째 요청: 이름 알려주기
history = [
    {
        "type": "user_input",
        "content": [
            {"type": "text", "text": "내 이름은 민수야. '확인'이라고만 답해."}
        ],
    }
]

turn1 = client.interactions.create(
    model=model, input=history, store=False,
)
print("첫 번째 답변:", turn1.output_text)

# 기존 user_input 뒤에 thought와 model_output을 추가합니다.
history.extend(
    step.model_dump(exclude_none=True) for step in turn1.steps
)

# 두 번째 요청: 이름 물어보기
history.append(
    {
        "type": "user_input",
        "content": [
            {"type": "text", "text": "내 이름이 뭐야? 이름만 답해."}
        ],
    }
)

print()
print("두 번째 요청에 보내는 대화 기록:")
pprint(history)

turn2 = client.interactions.create(
    model=model, input=history, store=False,
)
print("두 번째 답변:", turn2.output_text)

`step.model_dump(exclude_none=True)`는 SDK 객체 하나를 평범한 Python dictionary로 바꿉니다. `exclude_none=True`는 값이 `None`인 불필요한 항목을 결과에서 제외한다는 뜻입니다.

```python
step                    # SDK가 만든 Step 객체
step.model_dump()       # Step 객체 → Python dictionary
```

### `store=True`로 대화 이어가기

수동 history 방식은 우리 코드가 전체 대화 기록을 보관하고 매번 다시 보냅니다. Interactions API에서 `store=True`를 사용하면 서버가 interaction을 저장하고, 다음 요청에서 이전 interaction의 ID만 전달해 대화를 이어갈 수 있습니다.

- `store=True`: 이번 요청과 응답을 서버에 저장합니다.
- `interaction.id`: 저장된 interaction의 고유 ID입니다.
- `previous_interaction_id`: 다음 요청이 어느 대화 다음에 이어지는지 지정합니다.

> 실제 서비스에서는 개인정보를 보내기 전에 데이터 저장 정책과 보관 기간을 확인해야 합니다.

In [ ]:
# 첫 번째 요청을 서버에 저장합니다.
stored_turn1 = client.interactions.create(
    model=model,
    input="내 이름은 민수야. '확인'이라고만 답해.",
    store=True,
)
print("첫 번째 답변:", stored_turn1.output_text)
print("interaction ID:", stored_turn1.id)

# 전체 history 대신 이전 interaction의 ID를 전달합니다.
stored_turn2 = client.interactions.create(
    model=model,
    input="내 이름이 뭐야? 이름만 답해.",
    previous_interaction_id=stored_turn1.id,
    store=True,
)
print("두 번째 답변:", stored_turn2.output_text)

### 이전 시점에서 새로운 대화로 분기하기

`previous_interaction_id`에는 가장 최근 ID만 넣어야 하는 것이 아닙니다. 이전 interaction의 ID를 다시 사용하면 그 시점까지의 맥락에서 새로운 대화를 시작할 수 있습니다.

아래 예제에서는 1번에서 이름을 알려주고, 2번에서 좋아하는 음식 정보를 추가합니다. 그런 다음 1번으로 돌아가 새 대화를 분기하면 2번에서 추가한 정보는 전달되지 않습니다. 기존의 2번 대화가 삭제되는 것은 아닙니다.

In [5]:
# 1. 이름 알려주기
branch_turn1 = client.interactions.create(
    model=model,
    input="내 이름은 민수야.",
    store=True,
)
print("1번:", branch_turn1.output_text)

# 2. 이름을 물어보고 새로운 정보 추가하기
branch_turn2 = client.interactions.create(
    model=model,
    input="내 이름이 뭐야? 그리고 내가 좋아하는 음식은 피자야. 두 내용을 확인해줘.",
    previous_interaction_id=branch_turn1.id,
    store=True,
)
print("2번:", branch_turn2.output_text)

# 3. 2번이 아닌 1번에서 분기해 좋아하는 음식 물어보기
branch_from_turn1 = client.interactions.create(
    model=model,
    input="내 이름과 내가 좋아하는 음식이 뭐야?",
    previous_interaction_id=branch_turn1.id,
    store=True,
)
print("1번에서 새로 분기:", branch_from_turn1.output_text)

1번: 안녕하세요, 민수님! 반갑습니다. 오늘 어떤 도움이 필요하신가요? 편하게 말씀해 주세요!
2번: 네, 확인해 드릴게요!

1. **이름:** 민수
2. **좋아하는 음식:** 피자

두 가지 내용을 잘 기억하고 있겠습니다! 피자를 정말 좋아하시는군요. 특별히 좋아하는 피자 브랜드나 종류(포테이토, 페퍼로니 등)가 있으신가요?
1번에서 새로 분기: 민수님의 이름은 **민수**입니다! 

하지만 좋아하는 음식은 아직 저에게 말씀해 주시지 않아서 모르고 있어요. 민수님은 어떤 음식을 가장 좋아하시나요? 알려주시면 기억해 둘게요!


### 대화 맥락의 핵심

- history 없이 물으면 이름 정보가 없으므로 `모름`이라고 답합니다.
- 이름이 들어 있는 history를 함께 보내면 `민수`라고 답합니다.
- 모델 자체가 개인을 영구적으로 기억하는 것은 아닙니다.
- 수동 history 방식에서는 앱이 `history` 전체를 매번 다시 보냅니다.
- `store=True` 방식에서는 서버가 대화를 저장하고 ID로 연결합니다.
- 이전 interaction의 ID를 다시 지정하면 그 시점에서 새로운 대화로 분기할 수 있습니다.
- 대화가 길어지면 토큰, 비용과 지연 시간이 늘 수 있습니다.
- 실제 서비스에서는 대화 기록의 길이, 개인정보와 서버 저장 정책을 관리해야 합니다.

## Generation config

프롬프트는 모델에게 원하는 답변 방식을 설명하고, generation_config는 답변을 생성하는 방법을 코드로 설정합니다.

- `max_output_tokens`: 생성할 수 있는 최대 token 수입니다. 정확한 글자 수가 아니며, 너무 작으면 답변이 중간에 끝날 수 있습니다.
- `thinking_level`: 모델이 답변 전에 사용하는 추론 수준입니다. `low`, `medium`, `high` 중에서 선택합니다.

> 같은 설정이어도 답변이 항상 완전히 같아지는 것은 아닙니다.

In [ ]:
configured = client.interactions.create(
    model=model,
    input="LLM API를 한 문장으로 설명해줘.",
    generation_config={
        "max_output_tokens": 30,
    },
    store=False,
)
print(configured.output_text)

## 실패도 정상적인 흐름이다

API 호출 실패를 모두 같은 오류로 처리하면 원인과 대응 방법을 알 수 없습니다. Google GenAI SDK는 400번대 응답을 `ClientError`, 500번대 응답을 `ServerError`로 구분합니다.

| 오류 | 예 | 기본 대응 |
|---|---|---|
| `ClientError` | 잘못된 API 키, 모델 이름, 요청 형식 | 요청이나 설정을 수정해야 합니다. |
| `ClientError` 429 | 사용량 제한 | 사용량과 제한을 확인하고 잠시 후 다시 요청합니다. |
| `ServerError` | Gemini 서버의 일시적인 500번대 오류 | 잠시 후 다시 요청합니다. |

In [ ]:
def ask_gemini(prompt: str) -> str | None:
    try:
        result = client.interactions.create(
            model=model, input=prompt, store=False,
        )
        return result.output_text
    except errors.ClientError as error:
        if error.code in [401, 403]:
            print("인증 실패: API 키와 권한을 확인하세요.")
        elif error.code == 404:
            print("요청한 모델을 찾을 수 없습니다.")
        elif error.code == 429:
            print("사용량 제한에 도달했습니다. 잠시 후 다시 요청하세요.")
        else:
            print(f"요청 오류({error.code}): {error.message}")
        return None
    except errors.ServerError as error:
        print(f"서버 오류({error.code}): 잠시 후 다시 요청하세요.")
        return None

answer = ask_gemini("Python 함수를 30자 이내의 한 문장으로 설명해줘.")
if answer:
    print(answer)

## 종합 실습: 대화형 여행 플래너 만들기

앞에서 배운 기능을 조합해 대화형 여행 플래너를 만들어 보세요.

여행지를 입력받아 첫 여행 일정을 만든 뒤, 사용자가 원하는 만큼 추가 요청이나 수정 의견을 반영해 보세요. 앞에서 나눈 내용을 기억한 상태로 답해야 하며, 사용자가 `종료`를 입력하면 여행 계획을 마칩니다.

In [11]:
instruction =  "여행 계획 전문가야 사용자의 여행 계획을 만든 뒤 친절하게 계획을 알려준다. 사용자가 종료라고 입력하면 계획을 마치고 알려준다" 

previous_id = None

while True :
    user_input = input("요청사항 입력/ 종료시 종료 입력")

    stored_turn1 = client.interactions.create(
        model=model,
        input=user_input,
        store=True,
        system_instruction=instruction,
        previous_interaction_id=previous_id
    )
    previous_id = stored_turn1.id
    print(stored_turn1.output_text)

    if user_input == "종료":
        print("여행 계획 종료")
        break 


안녕하세요! 당신만의 멋진 여행을 만들어 드릴 **여행 계획 전문가**입니다. 😊

어디로 떠나고 싶으신가요? 완벽한 여행 계획을 세워드릴 수 있도록 아래 내용을 편하게 말씀해 주세요!

1. **목적지** (예: 제주도, 도쿄, 파리 등)
2. **여행 기간 및 일정** (예: 2박 3일, 10월 중순 등)
3. **여행 스타일** (예: 맛집 탐방, 휴양, 액티비티, 문화재 관람 등)
4. **동행인** (예: 혼자, 연인, 가족, 친구 등)
5. **대략적인 예산** (선택 사항)

원하시는 정보를 자유롭게 알려주시면 최적의 일정을 짜드릴게요. 
*(여행 계획 작성을 마치고 싶으실 땐 언제든 **'종료'**라고 입력해 주세요!)*
여행 계획 작성을 종료합니다! 

아직 자세한 계획을 세우지 못해 아쉽지만, 다음에 여행을 떠나고 싶으실 때 언제든 다시 찾아주세요. 

늘 즐겁고 행복한 일상 되시길 바랍니다! ✈️🧳✨
여행 계획 종료


In [ ]:
# 여행지를 입력 받아서 여행 일정을 만들기
#   llm api에 요청을 보내야 해.

# 첫번째 일정 -> 추가요청/수정의견 -> 두번째 일정 -> 추가요청/수정의견 -> 
# 첫번째 : 이전 요청 id가 필요 없음 / 두번째부터 필요함.
system_instruction = """
너는 여행 플래너야.
사용자의 요청에 따라서 여행 일정을 간단하게 짜줘.
3문장 이내로 답변해.
"""

user_input = input()
interaction = client.interactions.create(
    model=model,
    input=user_input,
    system_instruction=system_instruction,
    store=True,
    generation_config={
        'max_output_tokens': 3000
    }
)
print(interaction.output_text)

while True:
    user_input = input().strip()

    if not user_input:
        break

    if user_input == "끝":
        break

    interaction = client.interactions.create(
        model=model,
        input=user_input,
        system_instruction=system_instruction,

        # 대화가 이어갈 수 있도록 해준다.
        previous_interaction_id=interaction.id,
        store=True,
    )

    print(interaction.output_text)

